<a href="https://colab.research.google.com/github/noencrp87/labo3-2026ba/blob/main/316_AutoGluon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoGluon
Matar una mosca con una  bazooka, entrena automaticamente modelos de:


*   Estadistica Clasica
*   Machine Learning ~ GBDT
*   Deep Learning


 ['SeasonalNaive', 'RecursiveTabular', 'DirectTabular', 'DynamicOptimizedTheta', 'Chronos2', 'Chronos2SmallFineTuned', 'AutoETS', 'ChronosWithRegressor[bolt_small]', 'TemporalFusionTransformer', 'DeepAR']



![Matar una mosca con una bazooka ](https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/Bazooka_fly.jpg)

## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json


In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"


# 1  Modelo AutoGluon

## 1.1 Init Experimento

In [ ]:
# instalacion de paquetes que NO vienen por default en Colab
# autogluon es MUY pesado. varios minutos se perderan aqui
!pip install uv
!uv pip install -q kaggle
!uv pip install autogluon[all]


In [ ]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


In [ ]:
import os as os
import numpy as np
import polars as pl

from datetime import datetime
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

Por favor, cargar aqui SU semilla primigenia
<br> **Muy importante**, cambiar el numero de experimento en cada corrida. Usted ha sido notificado !
<br> Si cada corrida no está en una nueva carpeta virgen, entonces se reutilizarn modelos viejos corridos con otros parametros
https://www.youtube.com/shorts/0_0Kzqpdn1o

In [ ]:
# defino los parametros
PARAM = {'experimento':'AutoGluon-01',
  'kaggle_competition':'labo-iii-2026-ba',
  'semilla_primigenia':102191
}

In [ ]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

## 1.2 Init AutoGluon

In [ ]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [ ]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [ ]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [ ]:
# paso de periodo a  timestamp
tb_ventas = tb_ventas.with_columns(
    (pl.col('periodo').cast(pl.String).str.to_datetime('%Y%m')).alias('timestamp')
)

Opcion de *empiojar el dataset*
<br>agregando ruido relativo a las ventas
<br>Un Experimento no se le niega a nadie

In [ ]:
empiojar= False
empiojar_ruido= 0.25

if empiojar:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=empiojar_ruido, size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


## 1.3 Entrenamiento AutoGluon

La magia del Auto Machine Learning  aplicada a un dataset que posee MULTIPLES series de tiempo que se analizan en forma conjunta, suponiendo algun tipo de correlacion entre ellas.

AutoGluon TimeSeriesPredictor predicts future values of **multiple related** time series.

TimeSeriesPredictor provides probabilistic (quantile) multi-step-ahead orecasts for univariate time series. The forecast includes both the mean (i.e.,
 onditional expectation of future values given the past), as well as the quantiles of the forecast distribution, indicating the range of possible future outcomes.

TimeSeriesPredictor fits both “global” deep learning models that are **shared across all time series** (e.g., DeepAR, Transformer), as well as “local”
 statistical models that are fit to each individual time series (e.g., ARIMA, ETS).

TimeSeriesPredictor expects input data and makes predictions in the TimeSeriesDataFrame format.


https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.html

In [ ]:
# paso a formato  ts = time_series
ts_data = TimeSeriesDataFrame.from_data_frame(
  tb_ventas.to_pandas(), # tristemente AutoGluon espera un dataframe Pandas ...
  timestamp_column='timestamp',
  id_column='product_id'
)

Elegir cuidadosamente la metrica a utilizar
<br>Probar alternativas !
<br>Ese celda lleva 30 minutos en correr
<br>https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-metrics.html#forecasting-metrics
<br>https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.fit.html#autogluon.timeseries.TimeSeriesPredictor.fit
<br>https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.html

In [ ]:
# Entrenamiento, esta es la parte pesada
global_eval_metric = 'RMSE' # alternativa RMSE

# defino
modelo = TimeSeriesPredictor(
  prediction_length= 2,  # horizonte de prediccion
  target= 'tn',
  freq= 'MS',  # Frecuencia mensual (Month Start)
  eval_metric= global_eval_metric
)

# entreno, le tira con muchisimos algorimtos, y prueba ensamblarlos
modelo.fit(ts_data,
  num_val_windows= 2,
  time_limit= 3600, # una hora
  presets= "best_quality",  # Máxima calidad posible, obvio mayor tiempo de corrida
  random_seed= PARAM['semilla_primigenia']
)

## 1.4 Prediccion AutoGluon

In [ ]:
# predict a partir los mismos datos

tb_forecast = modelo.predict(ts_data,
  random_seed= PARAM['semilla_primigenia']
)

display(tb_forecast)

In [ ]:
# paso a formato Polars, teniendo en cuenta el indice
tb_forecast = pl.from_pandas(tb_forecast.reset_index())

In [ ]:
# me quedo con ls predicciones de febrero-2020
# en 'mean' esta la prediccion, mas alla de los n-tiles
tb_final = tb_forecast.filter(pl.col("timestamp") ==  datetime(2020, 2, 1)).select(["item_id","mean"])

display(tb_final)

## 1.5 Submit a Kaggle

In [ ]:
# cambio nombre de campos a los que reconoce Kaggle
tb_final = tb_final.rename({
  "item_id": "product_id",
  "mean": "tn",
})

In [ ]:
# Submit a Kaggle
if not empiojar:
  archivo= "AutoGluon_" + global_eval_metric + ".csv"
  mensaje= "AutoGluon " + global_eval_metric
else:
  archivo= "AutoGluon_empiojado_" + global_eval_metric + ".csv"
  mensaje= "AutoGluon logEMPIOJADO " + global_eval_metric + " al " + str(empiojar_ruido)

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )